In [80]:
#Our ML Modeling Notebook

In [81]:
#imports

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [82]:
#Upload dataset

from google.colab import files
upload = files.upload()

df = pd.read_csv('final_student_data.csv')

Saving final_student_data.csv to final_student_data (3).csv


# Final Preprocessing and Split

- Load the cleaned dataset

- separate features (X) and target (y) where the target is Dropout (1) vs Not Dropout (0)

- perform an 80/20 train/test split using random_state=42

- Make sure all features are ready for the models by ensuring categorical variables are properly encoded and scaling features if needed (especially for Logistic Regression). Your output should be X_train, X_test, y_train, y_test and a final modeling-ready dataset.

In [83]:
#separate features (X) and target (y) where the target is Dropout (1) vs Not Dropout (0)


#X = df['feature1', 'feature2', 'feature3']

X = df.drop('target', axis=1)

y = df['target']

In [84]:
# perform an 60/20/20 train/validation/test split using random_state=42
#splitting the test data from the rest

X_train_val, X_test, y_train_val, y_test = train_test_split(X,y,test_size=.20,random_state=42)

In [85]:
# splitting the training and validation data
X_train, X_val, y_train, y_val = train_test_split(X_train_val,y_train_val, test_size=.25,random_state= 42)

In [86]:
#scaling features
log_reg_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=42))
])

dt_pipeline = Pipeline([
    ('model', DecisionTreeClassifier(random_state=42))
])

rf_pipeline = Pipeline([
    ('model', RandomForestClassifier(random_state=42))
])

# Training Models

In [87]:
#training the all three models using the .fit() function from the sklearn.

log_reg_pipeline.fit(X_train, y_train)
dt_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

Pipeline(steps=[('model', RandomForestClassifier(random_state=42))])

In [88]:
#getting the predictions from each model

y_pred_log_reg = log_reg_pipeline.predict(X_val)
y_pred_dt = dt_pipeline.predict(X_val)
y_pred_rf = rf_pipeline.predict(X_val)

In [89]:
#getting the prediction probabilities

y_prob_log_reg = log_reg_pipeline.predict_proba(X_val) [:, 1]
y_prob_dt = dt_pipeline.predict_proba(X_val) [:, 1]
y_prob_rf = rf_pipeline.predict_proba(X_val) [:, 1]

In [90]:
#Our Notebook for Evaluating the ML Model

from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, f1_score

In [91]:
#Evaluating Scores for Logisitic
accuracy_log = accuracy_score(y_val, y_pred_log_reg)
precision_log = precision_score(y_val, y_pred_log_reg)
recall_log = recall_score(y_val, y_pred_log_reg)
f1_log = f1_score(y_val, y_pred_log_reg)

#Confusion Matrix for Logistic
cm_log_reg = confusion_matrix(y_val, y_pred_log_reg)

In [92]:
cm_log_reg

array([[569,  33],
       [ 87, 196]])

In [93]:
#Evaluating Scores for Decision
accuracy_dt = accuracy_score(y_val, y_pred_dt)
precision_dt = precision_score(y_val, y_pred_dt)
recall_dt = recall_score(y_val, y_pred_dt)
f1_dt = f1_score(y_val, y_pred_dt)

cm_dt = confusion_matrix(y_val, y_pred_dt)

In [94]:
cm_dt

array([[521,  81],
       [ 92, 191]])

In [95]:
#Evaluating Scores for Random Forest
accuracy_rf = accuracy_score(y_val, y_pred_rf)
precision_rf = precision_score(y_val, y_pred_rf)
recall_rf = recall_score(y_val, y_pred_rf)
f1_rf = f1_score(y_val, y_pred_rf)

cm_rf = confusion_matrix(y_val, y_pred_rf)

In [96]:
cm_rf

array([[573,  29],
       [ 92, 191]])

In [97]:
#Comparison Models
results_log_reg = pd.DataFrame([{
    "Model": "Logistic Regression:",
    "Accuracy": accuracy_log,
    "Precision": precision_log,
    "Recall": recall_log,
    "F1": f1_log
}])

results_dt = pd.DataFrame([{
    "Model": "Decision Tree:",
    "Accuracy": accuracy_dt,
    "Precision": precision_dt,
    "Recall": recall_dt,
    "F1": f1_dt
}])

results_rf = pd.DataFrame([{
    "Model": "Random Forest:",
    "Accuracy": accuracy_rf,
    "Precision": precision_rf,
    "Recall": recall_rf,
    "F1": f1_rf
}])

#Combine Results
results_df = pd.concat([results_log_reg, results_dt, results_rf], axis=0).reset_index(drop=True)
results_df

,Model,Accuracy,Precision,Recall,F1
0,Logistic Regression:,0.864407,0.855895,0.692580,0.765625
1,Decision Tree:,0.804520,0.702206,0.674912,0.688288
2,Random Forest:,0.863277,0.868182,0.674912,0.759443


# Optimization and Insights

In [98]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
import joblib

In [99]:
#tune logistic reg

from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear'],
    'class_weight': [None, 'balanced']
}

logreg_grid = GridSearchCV(LogisticRegression(max_iter=1000),
                    param_grid,
                    cv=5,
                    scoring='f1')

logreg_grid.fit(X_train, y_train)

print(logreg_grid.best_params_)
print(logreg_grid.best_score_)

{'C': 0.1, 'class_weight': None, 'penalty': 'l2', 'solver': 'liblinear'}
0.7968519127161169


In [100]:
best_logreg_model = logreg_grid.best_estimator_

y_val_pred_tuned = best_logreg_model.predict(X_val)

print("Tuned Logistic Regression Validation Results:")
print(classification_report(y_val, y_val_pred_tuned))

Tuned Logistic Regression Validation Results:
              precision    recall  f1-score   support

           0       0.87      0.95      0.91       602
           1       0.87      0.70      0.77       283

    accuracy                           0.87       885
   macro avg       0.87      0.82      0.84       885
weighted avg       0.87      0.87      0.87       885



In [101]:
#tune Random Forest

rf_param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 5, 10, 15],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}

rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

print("Best Random Forest Parameters:")
print(rf_grid.best_params_)

print("Best Cross-Validation F1 Score:")
print(rf_grid.best_score_)

Best Random Forest Parameters:
{'model__max_depth': 15, 'model__min_samples_leaf': 2, 'model__min_samples_split': 2, 'model__n_estimators': 200}
Best Cross-Validation F1 Score:
0.7794974353298703


In [102]:
best_rf_model = rf_grid.best_estimator_

y_val_pred_tuned = best_rf_model.predict(X_val)

print("Tuned Random Forest Validation Results:")
print(classification_report(y_val, y_val_pred_tuned))

Tuned Random Forest Validation Results:
              precision    recall  f1-score   support

           0       0.87      0.95      0.91       602
           1       0.86      0.69      0.77       283

    accuracy                           0.87       885
   macro avg       0.87      0.82      0.84       885
weighted avg       0.87      0.87      0.86       885



In [103]:
#FINAL TEST EVALUATION

final_models = {
    "Logistic Regression Tuned": best_logreg_model,
    "Decision Tree": dt_pipeline,
    "Random Forest Tuned": best_rf_model
}

final_results = []

for model_name, model in final_models.items():
    y_test_pred = model.predict(X_test)

    final_results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_test_pred),
        "Precision": precision_score(y_test, y_test_pred),
        "Recall": recall_score(y_test, y_test_pred),
        "F1": f1_score(y_test, y_test_pred)
    })

final_results_df = pd.DataFrame(final_results)
final_results_df

,Model,Accuracy,Precision,Recall,F1
0,Logistic Regression Tuned,0.850847,0.859375,0.696203,0.769231
1,Decision Tree,0.772881,0.692308,0.655063,0.673171
2,Random Forest Tuned,0.853107,0.869048,0.693038,0.771127


In [104]:
#final classification report for best model (random forest won)

y_test_pred_final = best_rf_model.predict(X_test)
y_test_prob_final = best_rf_model.predict_proba(X_test)[:, 1]

print("Final Tuned Random Forest Test Results:")
print(classification_report(y_test, y_test_pred_final))

Final Tuned Random Forest Test Results:
              precision    recall  f1-score   support

           0       0.85      0.94      0.89       569
           1       0.87      0.69      0.77       316

    accuracy                           0.85       885
   macro avg       0.86      0.82      0.83       885
weighted avg       0.85      0.85      0.85       885



In [105]:
#final confusion matrix

final_cm = confusion_matrix(y_test, y_test_pred_final)

print("Final Tuned Random Forest Confusion Matrix:")
print(final_cm)

Final Tuned Random Forest Confusion Matrix:
[[536  33]
 [ 97 219]]


In [106]:
#export final predictions

final_predictions = X_test.copy()
final_predictions["actual_dropout"] = y_test.values
final_predictions["predicted_dropout"] = y_test_pred_final
final_predictions["dropout_probability"] = y_test_prob_final

final_predictions.to_csv("final_model_predictions.csv", index=False)

final_predictions.head()

,marital_status,application_order,daytime_evening_attendance,previous_qualification,previous_qualification_grade,admission_grade,displaced,educational_special_needs,debtor,tuition_fees_up_to_date,...,course_Health,course_Media/Communication,course_Social Services,application_mode_Other,application_mode_Regular Admission,application_mode_Special Entry,application_mode_Transfer/Change,actual_dropout,predicted_dropout,dropout_probability
1255,4,1,1,1,133.1,110.0,1,0,0,1,...,0,0,0,0,0,0,0,1,1,0.825901
3458,1,1,1,1,125.0,119.8,0,0,0,1,...,0,0,1,0,1,0,0,0,0,0.065545
3390,1,1,1,1,133.0,127.4,0,0,1,1,...,0,0,0,0,1,0,0,0,0,0.112608
1497,1,2,1,1,110.0,115.3,1,0,0,1,...,0,0,0,0,1,0,0,0,0,0.105014
1536,1,1,1,1,130.0,106.2,1,0,0,1,...,1,0,0,0,0,0,0,1,1,0.922351


In [107]:
#feature importance table

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": best_rf_model.named_steps["model"].feature_importances_
}).sort_values(by="Importance", ascending=False)

feature_importance.to_csv("feature_importance.csv", index=False)

feature_importance.head(10)

,Feature,Importance
23,curricular_units_2nd_sem_approved,0.164064
24,curricular_units_2nd_sem_grade,0.131337
17,curricular_units_1st_sem_approved,0.100618
18,curricular_units_1st_sem_grade,0.079356
9,tuition_fees_up_to_date,0.065248
12,age_at_enrollment,0.046036
22,curricular_units_2nd_sem_evaluations,0.042500
16,curricular_units_1st_sem_evaluations,0.033003
5,admission_grade,0.032270
4,previous_qualification_grade,0.029483


In [108]:
#save final tuned model

joblib.dump(best_rf_model, "final_random_forest_model.pkl")

['final_random_forest_model.pkl']

In [109]:
#export final model comparison table

final_results_df.to_csv("final_model_comparison.csv", index=False)